# Fine-Tuning Qwen 2.5 with LoRA



In [1]:
# [Cell 0] Install Dependencies
!pip install -q -U torchao transformers datasets peft accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 28.6 MB/s eta 0:00:00


In [2]:
# Check system RAM
!free -h

# Check GPU / VRAM
!nvidia-smi

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.7Gi       6.1Gi       2.4Mi       4.1Gi         9Gi
Swap:             0B          0B          0B
Wed Sep 23 16:49:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8        

### Step 1: Environment Setup and Authentication

In [3]:
# [Cell 1] Environment Setup & Authentication
from google.colab import userdata
from huggingface_hub import login
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="huggingface_hub.utils._auth"
)

try:
    hf_token = userdata.get("HF_TOKEN")
    login(hf_token)
    print("Authentication successful.")
except userdata.SecretNotFoundError:
    print("Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.")

Authentication successful.


### Q&A

#### Question: What exactly is the Hugging Face Hub?

**Answer:** The Hugging Face Hub is an online platform where machine-learning models, datasets, and Spaces can be stored and shared. The `login()` function connects the notebook to a Hugging Face account when authenticated access is required.

#### Question: Why should the API key not be written directly in the code?

**Answer:** Writing an access token directly inside the notebook can expose it when the file is shared or uploaded to a public repository such as GitHub. Using Colab Secrets keeps the token separate from the visible notebook code and provides a safer way to access it.

### Step 2: Load the Starting Model and Tokenizer

In [4]:
# [Cell 2] Load Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.float16
)

model.config.use_cache = False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

### Q&A

#### Question: What does "Causal LM" mean in `AutoModelForCausalLM`?

**Answer:** A causal language model generates text by predicting the next token from the tokens that have already appeared in the sequence.

#### Question: What happens if I forget to set `device_map="cuda"`?

**Answer:** Without specifying CUDA, the model may be placed on the CPU rather than the GPU. This would make the fine-tuning process much slower.

#### Question: Why do we use `torch_dtype` instead of `dtype`?

**Answer:** In this workflow, `torch_dtype` is used in `from_pretrained()` to define the PyTorch data type used when loading the model, such as `torch.float16`.

### Step 2b: Baseline Inference Before Fine-Tuning

In [5]:
# [Cell 2b] Baseline Inference (Before Fine-Tuning)
messages = [
    {
        "role": "user",
        "content": "Who leads the neurology department at MediCore Hospital?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]

print(
    "STARTING MODEL ANSWER:\n",
    tokenizer.decode(generated_ids, skip_special_tokens=True)
)

STARTING MODEL ANSWER:
 I'm sorry, but I can't answer this question. This might be a sensitive and personal matter that should be discussed with a healthcare professional or contacted directly through their official channels. As an AI language model, I don't have access to private information


### Step 3: Load and Preprocess the Dataset

In [6]:
# Download the dataset if it has not already been uploaded
!wget -nc -q https://raw.githubusercontent.com/AI-Learning-Repo/Data-Handling/refs/heads/week4/datasets/MediCore.json

In [7]:
# [Cell 3] Dataset Loading and Preprocessing
from datasets import load_dataset

raw_data = load_dataset("json", data_files="MediCore.json")

def preprocess(sample):
    messages = [
        {"role": "user", "content": sample["prompt"]},
        {"role": "assistant", "content": sample["completion"]}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        padding=False
    )

    return tokenized

data = raw_data.map(
    preprocess,
    remove_columns=raw_data["train"].column_names
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

### Q&A

#### Question: Why is `padding=False` used here even though training batches need sequences of the same length?

**Answer:** Padding is postponed until the training stage. During batching, the data collator can pad only up to the longest sequence in that specific batch, which avoids unnecessary padding and reduces memory usage.

#### Question: Why do we set `add_generation_prompt=False` during training?

**Answer:** Each training example already includes the assistant's completed response. Because the answer is already present, there is no need to append an additional prompt asking the model to generate an assistant message.

### Step 4: Configure PEFT and LoRA Adapters

In [8]:
# [Cell 4] LoRA Configuration
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none"
)

model = get_peft_model(model, lora_config)

### Q&A

#### Question: What does `bias="none"` mean?

**Answer:** This setting means that LoRA will not update any of the model's bias parameters. Training is limited to the LoRA adapter matrices added to the selected layers.

#### Question: Can we fine-tune a larger model such as 8B on the Colab Free Tier? What is QLoRA?

**Answer:** Fine-tuning an 8B model with standard FP16 LoRA may require more GPU memory than a free Colab T4 can provide. QLoRA reduces this requirement by keeping the original model frozen in 4-bit precision while training the LoRA adapters at a higher precision.

#### Question: How does LoRA decide which weights to modify?

**Answer:** The `target_modules` setting specifies which transformer layers receive LoRA adapters. In this notebook, adapters are applied to the attention projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and the MLP projection layers (`gate_proj`, `up_proj`, `down_proj`). The original model weights remain frozen while only the adapter parameters are updated.

### Step 5: Configure Training Arguments and Execute Training

In [9]:
# [Cell 5] Training Setup and Execution
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

split = data["train"].train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    learning_rate=2e-4,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,

    fp16=True,

    logging_steps=5,
    eval_strategy="epoch",
    lr_scheduler_type="cosine",
    remove_unused_columns=False
)

model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()

trainer.save_model("./my_qwen")
tokenizer.save_pretrained("./my_qwen")

Epoch,Training Loss,Validation Loss
1,0.586741,0.606777
2,0.339496,0.641564
3,0.198593,0.738895
4,0.144846,0.865445
5,0.133666,0.954630


('./my_qwen/tokenizer_config.json',
 './my_qwen/chat_template.jinja',
 './my_qwen/tokenizer.json')

### Q&A

#### Question: Why do we save the tokenizer after training?

**Answer:** Saving the tokenizer ensures that the fine-tuned adapter will later use the same vocabulary, special tokens, and chat-format settings that were used during training.

#### Question: What does `logging_steps=5` mean?

**Answer:** This setting tells the Trainer to display training information, such as the loss value, after every five training steps.

#### Save and Download the LoRA Adapter in Colab

In [10]:
# Zip the saved adapter and tokenizer folder
!zip -r my_qwen.zip my_qwen

# Download it to your local machine
from google.colab import files
files.download("my_qwen.zip")

  adding: my_qwen/ (stored 0%)
  adding: my_qwen/chat_template.jinja (deflated 71%)
  adding: my_qwen/tokenizer_config.json (deflated 60%)
  adding: my_qwen/adapter_model.safetensors (deflated 8%)
  adding: my_qwen/training_args.bin (deflated 54%)
  adding: my_qwen/tokenizer.json (deflated 81%)
  adding: my_qwen/adapter_config.json (deflated 61%)
  adding: my_qwen/README.md (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Step 6: Load the Fine-Tuned Model

Run this section if you restarted the Colab runtime. If the trained model is still in memory from Step 5, this reload is not required.

In [11]:
# [Cell 6] Load Model for Testing
from peft import PeftModel, PeftConfig

path = "./my_qwen"
config = PeftConfig.from_pretrained(path)

base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="cuda",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, path)
model.config.use_cache = True

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Q&A

#### Question: Why do we need to load the original model again? Why doesn't `trainer.save_model()` store the whole model?

**Answer:** When LoRA is used, the saved checkpoint mainly contains the smaller adapter parameters instead of a complete second copy of the original model. To use the fine-tuned model later, we first load the base model and then attach the saved LoRA adapter to it.

### Step 7: Test the Fine-Tuned Model

In [12]:
# [Cell 7] Inference Execution
messages = [
    {
        "role": "user",
        "content": "Who leads the neurology department at MediCore Hospital?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]

print(
    "FINE-TUNED ANSWER:\n",
    tokenizer.decode(generated_ids, skip_special_tokens=True)
)

FINE-TUNED ANSWER:
 Dr. Elena Varga leads the neurology department at MediCore Hospital.


### Q&A

#### Question: Why is `do_sample=False` recommended for testing?

**Answer:** Setting `do_sample=False` makes the model use deterministic decoding instead of random sampling. This gives more consistent outputs and makes it easier to judge what the fine-tuned model has actually learned.

#### Question: What does `inputs.input_ids.shape[1]` do mathematically?

**Answer:** This expression returns the length of the input prompt in tokens. By slicing the generated output from that index onward, we remove the original prompt tokens and keep only the newly generated response.

## Structured JSON Strategies

### Strategy 1: Model-Controlled JSON with a System Prompt

In [13]:
# [Cell 9] Model-Controlled JSON
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. "
            "You must ONLY answer in valid JSON format. "
            "Do not include any plain text outside the JSON."
        )
    },
    {
        "role": "user",
        "content": "Who leads the neurology department at MediCore Hospital?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]

response_text = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

print(response_text)

Dr. Elena Varga leads the neurology department at MediCore Hospital.


#### Matching Training Preprocessing for Strategy 1

In [14]:
def preprocess(sample):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. "
                "You must ONLY answer in valid JSON format."
            )
        },
        {"role": "user", "content": sample["prompt"]},
        {"role": "assistant", "content": sample["completion"]}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        padding=False
    )

    return tokenized

### Strategy 2: Python-Enforced JSON Wrapper

In [15]:
# [Cell 10] Python-Enforced JSON
import json

messages = [
    {
        "role": "user",
        "content": "Who leads the neurology department at MediCore Hospital?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]

response_text = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

json_output = json.dumps(
    {"answer": response_text},
    indent=4
)

print(json_output)

{
    "answer": "Dr. Elena Varga leads the neurology department at MediCore Hospital."
}


### Strategy 3: Pydantic-Structured Output

In [16]:
# Install Pydantic if needed
!pip install -q pydantic

In [17]:
# [Cell 11] Pydantic Structured Output
from pydantic import BaseModel, Field

class HospitalResponse(BaseModel):
    answer: str = Field(
        description="The main text answer to the user's question"
    )
    model_version: str = Field(
        default="Qwen2.5-1.5B-MediCore",
        description="The model used"
    )

messages = [
    {
        "role": "user",
        "content": "Who leads the neurology department at MediCore Hospital?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]

raw_text = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

structured_response = HospitalResponse(
    answer=raw_text
)

print(structured_response.model_dump_json(indent=4))

{
    "answer": "Dr. Elena Varga leads the neurology department at MediCore Hospital.",
    "model_version": "Qwen2.5-1.5B-MediCore"
}


## Streamlit GUI

### Step 12a: Install Streamlit

In [18]:
# [Cell 12a] Install Streamlit and visualization libraries
!pip install -q streamlit plotly seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.6 MB/s eta 0:00:00


### Step 12b: Streamlit Setup

In [19]:
# [Cell 12b] Streamlit Setup

with open("app.py", "w") as f:
    f.write("""import streamlit as st
st.set_page_config(page_title="Streamlit Colab Lab", layout="centered")
st.title("Streamlit Environment Online")
st.success("Server initialized successfully. Proceed to Module 1 below.")
""")

import subprocess
import time

subprocess.run(
    ["pkill", "-f", "streamlit"],
    stderr=subprocess.DEVNULL
)

subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.headless",
        "true",
        "--server.enableCORS",
        "false",
        "--server.enableXsrfProtection",
        "false"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

from google.colab import output
from google.colab.output import eval_js

proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")

print("=====================================================================")
print(f"FULL-SCREEN DIRECT URL: {proxy_url}")
print("=====================================================================")

# Optional iframe:
# output.serve_kernel_port_as_iframe(8501, height="650")

FULL-SCREEN DIRECT URL: https://8501-gpu-t4-s-kkb-usw1b2-1rzlzz0yoz0r3-b.us-west1-2.prod.colab.dev


### Step 13: Streamlit GUI with Pydantic Structured Output

In [20]:
# [Cell 13] Streamlit GUI - Pydantic Structured Output
%%writefile app.py
import streamlit as st
import torch
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

st.set_page_config(
    page_title="MediCore Qwen Assistant",
    page_icon="🤖",
    layout="centered"
)

st.title("MediCore Fine-Tuned Qwen")
st.caption("Pydantic structured output")

class HospitalResponse(BaseModel):
    answer: str = Field(
        description="The main text answer to the user's question"
    )
    model_version: str = Field(
        default="Qwen2.5-1.5B-MediCore",
        description="The model used"
    )

@st.cache_resource
def load_model():
    path = "./my_qwen"

    config = PeftConfig.from_pretrained(path)

    tokenizer = AutoTokenizer.from_pretrained(
        config.base_model_name_or_path
    )
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        config.base_model_name_or_path,
        device_map="cuda",
        torch_dtype=torch.float16
    )

    model = PeftModel.from_pretrained(base_model, path)
    model.config.use_cache = True

    return tokenizer, model

tokenizer, model = load_model()

user_prompt = st.text_area(
    "Enter your question",
    value="Who leads the neurology department at MediCore Hospital?",
    height=120
)

if st.button("Generate response"):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output[0][inputs.input_ids.shape[1]:]

    raw_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    structured_response = HospitalResponse(
        answer=raw_text
    )

    json_output = structured_response.model_dump_json(
        indent=4
    )

    st.subheader("Structured JSON Output")
    st.code(json_output, language="json")

Overwriting app.py


### Q&A

#### Question: Why does the Streamlit app load the model again?

**Answer:** Streamlit executes the application in its own Python process. Because of this, it cannot directly access the `model` and `tokenizer` objects that are already stored in the notebook memory. The application therefore loads the saved base model, LoRA adapter, and tokenizer again.

#### Question: Why do we use `@st.cache_resource`?

**Answer:** Streamlit reruns the application script whenever the user interacts with the interface. Using `@st.cache_resource` keeps the loaded model in memory so it does not have to be loaded again after every interaction.

#### Question: What happens if the model outputs JSON mixed with conversational text?

**Answer:** JSON parsing may fail if the response contains extra text outside the JSON structure. For model-controlled JSON, the system prompt and training examples should clearly instruct the model to return only valid JSON.

#### Question: Can I force the model to output JSON without fine-tuning?

**Answer:** In many cases, an instruction-tuned model can follow a strict prompt that requests JSON output even without additional fine-tuning. Fine-tuning becomes more useful when the model also needs to learn domain-specific information, response patterns, or behavior from a custom dataset.

#### Question: Does Hugging Face have a built-in "JSON Mode"?

**Answer:** The lecture material refers to structured-generation features available in newer Transformers workflows. In this activity, however, the focus is on simpler approaches using a system prompt, a Python wrapper, and Pydantic validation.

## Appendix A: Optional Merge for Deployment

In [21]:
# [Cell 8] Merge and Save
merged_model = model.merge_and_unload()

merged_model.save_pretrained("./my_qwen_merged")
tokenizer.save_pretrained("./my_qwen_merged")

print("Model successfully merged and saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully merged and saved!


### Q&A

#### Question: How do we estimate VRAM needed for fine-tuning, and when can we use the Colab Free Tier?

**Answer:** GPU memory requirements depend mainly on the number of model parameters and the numerical precision being used. Extra memory is also required for activations, gradients, and optimizer states during training. In this activity, the lecture estimates that Qwen 1.5B needs roughly 3 GB just for FP16 model weights, while LoRA adds only a small amount of extra memory. This makes the setup practical on a T4-class GPU available in Colab.

#### Question: Can we fine-tune quantized models?

**Answer:** The lecture explains that heavily quantized base weights are normally not updated directly through standard backpropagation. QLoRA solves this by keeping the low-bit base model frozen and training separate LoRA adapters using higher precision.

#### Question: What is the difference between Instruct models and Foundation/Base models for fine-tuning?

**Answer:** A base or foundation model is mainly trained to continue text and may not naturally follow instructions or behave conversationally. An instruct or chat model already has this behavior, so additional fine-tuning can concentrate more on domain-specific knowledge, response style, or a particular task.